In [3]:
# ============================================================
# 1. PRS Input Configuration
# ============================================================

from pathlib import Path
import pandas as pd

# Project directory
PROJECT_ROOT = Path.cwd().resolve()

# Demo genotype
GENOTYPE_PREFIX = (
    PROJECT_ROOT
    / "Demo_data/1-3.Sample_genotype/demo_chr22"
)

# External GWAS and LD reference
EXTERNAL_DIR = PROJECT_ROOT / "External_data"

GWAS_FILES = {
    ancestry: EXTERNAL_DIR / "GWAS" / f"DIAMANTE_{ancestry}.txt"
    for ancestry in ["EAS", "EUR", "SAS"]
}

LD_DIR = EXTERNAL_DIR / "LD_reference"

SNPINFO_FILE = LD_DIR / "snpinfo_mult_1kg_hm3"

# Analysis settings
CHROM = 22
PHI = 1e-2
SEED = 99999

# Output directory
OUTPUT_DIR = PROJECT_ROOT / "Outputs" / "Step1_PRS"

# Check required input files
inputs = {
    "Genotype BED": Path(str(GENOTYPE_PREFIX) + ".bed"),
    "Genotype BIM": Path(str(GENOTYPE_PREFIX) + ".bim"),
    "Genotype FAM": Path(str(GENOTYPE_PREFIX) + ".fam"),
    "GWAS EAS": GWAS_FILES["EAS"],
    "GWAS EUR": GWAS_FILES["EUR"],
    "GWAS SAS": GWAS_FILES["SAS"],
    "LD Reference": LD_DIR,
    "SNP Information": SNPINFO_FILE,
}

for name, path in inputs.items():
    print(f"{name:20s}: {'OK' if path.exists() else 'MISSING'}")

assert all(path.exists() for path in inputs.values()), (
    "Some PRS input files are missing."
)

print("\nChromosome:", CHROM)
print("JointPRS phi:", PHI)
print("Random seed:", SEED)
print("\nPRS input configuration ready.")

Genotype BED        : OK
Genotype BIM        : OK
Genotype FAM        : OK
GWAS EAS            : OK
GWAS EUR            : OK
GWAS SAS            : OK
LD Reference        : OK
SNP Information     : OK

Chromosome: 22
JointPRS phi: 0.01
Random seed: 99999

PRS input configuration ready.


In [4]:
# ============================================================
# 2. Inspect GWAS Summary Statistics
# ============================================================

ancestries = ["EAS", "EUR", "SAS"]

# Demo genotype SNP information
bim = pd.read_csv(
    str(GENOTYPE_PREFIX) + ".bim",
    sep=r"\s+",
    header=None,
    names=["CHR", "SNP", "CM", "BP", "A1", "A2"]
)

print(f"Demo genotype: {len(bim):,} SNPs")
print("Chromosomes:", sorted(bim["CHR"].unique().tolist()))

# GWAS input format
for ancestry in ancestries:
    print(f"\n{'=' * 50}")
    print(f"{ancestry} GWAS")
    print("=" * 50)

    gwas_preview = pd.read_csv(
        GWAS_FILES[ancestry],
        sep=r"\s+",
        nrows=5
    )

    print("Columns:", gwas_preview.columns.tolist())
    display(gwas_preview)

Demo genotype: 1,000 SNPs
Chromosomes: [22]

EAS GWAS
Columns: ['SNP', 'A1', 'A2', 'BETA', 'P']


,SNP,A1,A2,BETA,P
0,rs12238997,A,G,-0.0282,0.3987
1,rs55727773,A,G,-0.0095,0.7037
2,rs12565286,C,G,0.0049,0.8981
3,rs4951859,C,G,0.0042,0.8530
4,rs142557973,T,C,-0.0229,0.4777



EUR GWAS
Columns: ['SNP', 'A1', 'A2', 'BETA', 'P']


,SNP,A1,A2,BETA,P
0,rs12238997,A,G,0.0044,0.7124
1,rs55727773,A,G,0.0118,0.2355
2,rs12565286,C,G,-0.0176,0.5013
3,rs4951859,C,G,-0.0043,0.6813
4,rs148120343,T,C,0.0001,0.9954



SAS GWAS
Columns: ['SNP', 'A1', 'A2', 'BETA', 'P']


,SNP,A1,A2,BETA,P
0,rs12238997,A,G,0.0224,0.5062
1,rs55727773,A,G,0.0410,0.1589
2,rs12565286,C,G,0.0540,0.3883
3,rs4951859,C,G,0.0090,0.7530
4,rs148120343,T,C,0.1052,0.3296


In [6]:
# ============================================================
# 3. PRS-CSx: Posterior SNP Effect Estimation
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path

# ============================================================
# 3.1 User-defined parameters
# ============================================================

# Discovery populations (order must match GWAS sample sizes)
POPULATIONS = ["EAS", "EUR", "SAS"]

# GWAS sample sizes
GWAS_N = {
    "EAS": 283423,
    "EUR": 933970,
    "SAS": 49492,
}

# Input data
PRSCSX_REF_DIR = LD_DIR
PRSCSX_BIM_PREFIX = GENOTYPE_PREFIX
PRSCSX_GWAS_FILES = GWAS_FILES

# Chromosome
PRSCSX_CHROM = 22

# Global shrinkage parameter
# "auto": estimate phi using the fully Bayesian approach
# Alternatively, specify a numeric value such as 1e-2
PRSCSX_PHI = "auto"

# MCMC parameters
PRSCSX_N_ITER = 3000
PRSCSX_N_BURNIN = 1500
PRSCSX_THIN = 5

# Gamma-gamma prior parameters
PRSCSX_A = 1
PRSCSX_B = 0.5

# Meta-analysis of posterior SNP effects
PRSCSX_META = True

# Save all posterior samples (not needed for standard PRS scoring)
PRSCSX_WRITE_PST = False

# Random seed
PRSCSX_SEED = 20261102

# Software environment
PRSCSX_ENV_NAME = "t2d_prscsx"
PRSCSX_SCRIPT = EXTERNAL_DIR / "Software/PRScsx/PRScsx.py"

# Output settings
PRSCSX_OUTPUT = OUTPUT_DIR / "PRSCSx" / "auto_meta"
PRSCSX_OUT_NAME = "Demo_PRSCSx"

# Number of CPU threads for numerical libraries
PRSCSX_THREADS = 1


# ============================================================
# 3.2 Check inputs and software
# ============================================================

CONDA = os.environ.get("CONDA_EXE") or shutil.which("conda")

if CONDA is None:
    for candidate in [
        Path.home() / "miniforge3/bin/conda",
        Path.home() / "miniconda3/bin/conda",
    ]:
        if candidate.is_file():
            CONDA = str(candidate)
            break

if CONDA is None:
    raise FileNotFoundError("Conda executable not found.")

if not PRSCSX_SCRIPT.is_file():
    raise FileNotFoundError(PRSCSX_SCRIPT)

for population in POPULATIONS:
    if not PRSCSX_GWAS_FILES[population].is_file():
        raise FileNotFoundError(PRSCSX_GWAS_FILES[population])

if not PRSCSX_REF_DIR.is_dir():
    raise FileNotFoundError(PRSCSX_REF_DIR)

if not Path(str(PRSCSX_BIM_PREFIX) + ".bim").is_file():
    raise FileNotFoundError(str(PRSCSX_BIM_PREFIX) + ".bim")

assert PRSCSX_N_ITER > PRSCSX_N_BURNIN >= 0
assert PRSCSX_THIN > 0
assert PRSCSX_PHI == "auto" or float(PRSCSX_PHI) > 0

PRSCSX_OUTPUT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3.3 Build PRS-CSx command
# ============================================================

cmd = [
    str(CONDA),
    "run",
    "--no-capture-output",
    "-n", PRSCSX_ENV_NAME,
    "python",
    str(PRSCSX_SCRIPT),

    # Input
    f"--ref_dir={PRSCSX_REF_DIR}",
    f"--bim_prefix={PRSCSX_BIM_PREFIX}",

    "--sst_file=" + ",".join(
        str(PRSCSX_GWAS_FILES[p]) for p in POPULATIONS
    ),

    "--n_gwas=" + ",".join(
        str(GWAS_N[p]) for p in POPULATIONS
    ),

    "--pop=" + ",".join(POPULATIONS),

    # Model parameters
    f"--chrom={PRSCSX_CHROM}",
    f"--a={PRSCSX_A}",
    f"--b={PRSCSX_B}",

    # MCMC
    f"--n_iter={PRSCSX_N_ITER}",
    f"--n_burnin={PRSCSX_N_BURNIN}",
    f"--thin={PRSCSX_THIN}",

    # Meta-analysis
    f"--meta={str(PRSCSX_META)}",
    f"--write_pst={str(PRSCSX_WRITE_PST)}",

    # Reproducibility
    f"--seed={PRSCSX_SEED}",

    # Output
    f"--out_dir={PRSCSX_OUTPUT}",
    f"--out_name={PRSCSX_OUT_NAME}",
]

# For auto mode, do NOT pass --phi
if PRSCSX_PHI != "auto":
    cmd.append(f"--phi={PRSCSX_PHI}")


# ============================================================
# 3.4 Display analysis settings
# ============================================================

print("=" * 65)
print("PRS-CSx ANALYSIS CONFIGURATION")
print("=" * 65)

print("Populations:", ", ".join(POPULATIONS))
print("GWAS sample sizes:", GWAS_N)
print("LD reference:", PRSCSX_REF_DIR)
print("Target genotype:", PRSCSX_BIM_PREFIX)

print("\nModel parameters:")
print("Chromosome:", PRSCSX_CHROM)
print("Phi:", PRSCSX_PHI)
print("Prior a:", PRSCSX_A)
print("Prior b:", PRSCSX_B)
print("Meta-analysis:", PRSCSX_META)

print("\nMCMC parameters:")
print("Iterations:", PRSCSX_N_ITER)
print("Burn-in:", PRSCSX_N_BURNIN)
print("Thinning:", PRSCSX_THIN)
print("Seed:", PRSCSX_SEED)

print("\nOutput directory:", PRSCSX_OUTPUT)


# ============================================================
# 3.5 Run PRS-CSx
# ============================================================

run_env = os.environ.copy()

run_env.update({
    "OMP_NUM_THREADS": str(PRSCSX_THREADS),
    "MKL_NUM_THREADS": str(PRSCSX_THREADS),
    "OPENBLAS_NUM_THREADS": str(PRSCSX_THREADS),
    "NUMEXPR_NUM_THREADS": str(PRSCSX_THREADS),
})

print("\nRunning PRS-CSx...", flush=True)

subprocess.run(
    cmd,
    check=True,
    env=run_env
)

print("\nPRS-CSx completed.")


# ============================================================
# 3.6 Check posterior effect files
# ============================================================

if PRSCSX_PHI == "auto":
    phi_tag = "phiauto"
else:
    phi_tag = f"phi{float(PRSCSX_PHI):.0e}"

effect_files = {}

for population in POPULATIONS + (["META"] if PRSCSX_META else []):

    matches = sorted(
        PRSCSX_OUTPUT.glob(
            f"{PRSCSX_OUT_NAME}_{population}_pst_eff_*"
            f"_{phi_tag}_chr{PRSCSX_CHROM}.txt"
        )
    )

    assert len(matches) == 1, (
        f"Expected one effect file for {population}, "
        f"found {len(matches)}"
    )

    effect_files[population] = matches[0]

print("\n" + "=" * 65)
print("PRS-CSx POSTERIOR EFFECT FILES")
print("=" * 65)

for population, file in effect_files.items():
    print(f"{population:5s}: {file.name}")

print(f"\nTotal effect files: {len(effect_files)}")
print("All expected PRS-CSx outputs verified.")

PRS-CSx ANALYSIS CONFIGURATION
Populations: EAS, EUR, SAS
GWAS sample sizes: {'EAS': 283423, 'EUR': 933970, 'SAS': 49492}
LD reference: External_data/LD_reference
Target genotype: Demo_data/1-3.Sample_genotype/demo_chr22

Model parameters:
Chromosome: 22
Phi: auto
Prior a: 1
Prior b: 0.5
Meta-analysis: True

MCMC parameters:
Iterations: 3000
Burn-in: 1500
Thinning: 5
Seed: 20261102

Output directory: Outputs/Step1_PRS/PRSCSx/auto_meta

Running PRS-CSx...


--ref_dir=External_data/LD_reference
--bim_prefix=Demo_data/1-3.Sample_genotype/demo_chr22
--sst_file=['External_data/GWAS/DIAMANTE_EAS.txt', 'External_data/GWAS/DIAMANTE_EUR.txt', 'External_data/GWAS/DIAMANTE_SAS.txt']
--a=1.0
--b=0.5
--phi=None
--n_gwas=[283423, 933970, 49492]
--pop=['EAS', 'EUR', 'SAS']
--n_iter=3000
--n_burnin=1500
--thin=5
--out_dir=Outputs/Step1_PRS/PRSCSx/auto_meta
--out_name=Demo_PRSCSx
--chrom=['22']
--meta=TRUE
--write_pst=FALSE
--seed=20261102


*** 3 discovery populations detected ***

##### process chrom

In [7]:
# ============================================================
# 4. JointPRS: Posterior SNP Effect Estimation
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path
import numpy as np

# ============================================================
# 4.1 User-defined parameters
# ============================================================

# Discovery populations
JOINTPRS_POPULATIONS = ["EAS", "EUR", "SAS"]

# GWAS sample sizes
JOINTPRS_GWAS_N = {
    "EAS": 283423,
    "EUR": 933970,
    "SAS": 49492,
}

# Input data
JOINTPRS_REF_DIR = LD_DIR
JOINTPRS_BIM_PREFIX = GENOTYPE_PREFIX
JOINTPRS_GWAS_FILES = GWAS_FILES

# Chromosome
JOINTPRS_CHROM = 22

# Global shrinkage parameter
# "auto": estimated automatically (omit --phi)
# Numeric: fixed phi, e.g. 1e-6, 1e-4, 1e-2, 1
JOINTPRS_PHI = 1e-2

# Genetic correlation constraints
# One value per discovery population.
# 1 allows estimation of positive pairwise genetic correlations.
JOINTPRS_RHO_CONS = [1, 1, 1]

# MCMC parameters
JOINTPRS_N_ITER = 3000
JOINTPRS_N_BURNIN = 1500
JOINTPRS_THIN = 5

# Prior parameters
JOINTPRS_A = 1
JOINTPRS_B = 0.5

# Additional posterior-effect meta-analysis
# Set True to generate an additional META effect file.
JOINTPRS_META = True

# Random seed
JOINTPRS_SEED = 20261102

# Software environment
JOINTPRS_ENV_NAME = "t2d_jointprs"
JOINTPRS_SCRIPT = (
    EXTERNAL_DIR / "Software/JointPRS/JointPRS.py"
)

# Output settings
JOINTPRS_OUTPUT = OUTPUT_DIR / "JointPRS" / "phi1e-2"
JOINTPRS_OUT_NAME = "Demo_JointPRS"

# Numerical library threads
JOINTPRS_THREADS = 1


# ============================================================
# 4.2 Check inputs and software
# ============================================================

CONDA = os.environ.get("CONDA_EXE") or shutil.which("conda")

if CONDA is None:
    for candidate in [
        Path.home() / "miniforge3/bin/conda",
        Path.home() / "miniconda3/bin/conda",
    ]:
        if candidate.is_file():
            CONDA = str(candidate)
            break

if CONDA is None:
    raise FileNotFoundError("Conda executable not found.")

if not JOINTPRS_SCRIPT.is_file():
    raise FileNotFoundError(JOINTPRS_SCRIPT)

if not JOINTPRS_REF_DIR.is_dir():
    raise FileNotFoundError(JOINTPRS_REF_DIR)

for population in JOINTPRS_POPULATIONS:
    if not JOINTPRS_GWAS_FILES[population].is_file():
        raise FileNotFoundError(
            JOINTPRS_GWAS_FILES[population]
        )

if not Path(
    str(JOINTPRS_BIM_PREFIX) + ".bim"
).is_file():
    raise FileNotFoundError(
        str(JOINTPRS_BIM_PREFIX) + ".bim"
    )

assert len(JOINTPRS_RHO_CONS) == len(JOINTPRS_POPULATIONS)
assert all(0 <= x <= 1 for x in JOINTPRS_RHO_CONS)
assert JOINTPRS_N_ITER > JOINTPRS_N_BURNIN >= 0
assert JOINTPRS_THIN > 0
assert JOINTPRS_PHI == "auto" or float(JOINTPRS_PHI) > 0

JOINTPRS_OUTPUT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 4.3 Build JointPRS command
# ============================================================

cmd = [
    str(CONDA),
    "run",
    "--no-capture-output",
    "-n", JOINTPRS_ENV_NAME,
    "python",
    str(JOINTPRS_SCRIPT),

    # Input
    f"--ref_dir={JOINTPRS_REF_DIR}",
    f"--bim_prefix={JOINTPRS_BIM_PREFIX}",

    "--sst_file=" + ",".join(
        str(JOINTPRS_GWAS_FILES[p])
        for p in JOINTPRS_POPULATIONS
    ),

    "--n_gwas=" + ",".join(
        str(JOINTPRS_GWAS_N[p])
        for p in JOINTPRS_POPULATIONS
    ),

    "--pop=" + ",".join(JOINTPRS_POPULATIONS),

    # Genetic correlation constraints
    "--rho_cons=" + ",".join(
        str(x) for x in JOINTPRS_RHO_CONS
    ),

    # Model parameters
    f"--chrom={JOINTPRS_CHROM}",
    f"--a={JOINTPRS_A}",
    f"--b={JOINTPRS_B}",

    # MCMC
    f"--n_iter={JOINTPRS_N_ITER}",
    f"--n_burnin={JOINTPRS_N_BURNIN}",
    f"--thin={JOINTPRS_THIN}",

    # Additional posterior META effects
    f"--meta={str(JOINTPRS_META)}",

    # Reproducibility
    f"--seed={JOINTPRS_SEED}",

    # Output
    f"--out_dir={JOINTPRS_OUTPUT}",
    f"--out_name={JOINTPRS_OUT_NAME}",
]

# Fixed phi versus automatic phi
if JOINTPRS_PHI != "auto":
    cmd.append(f"--phi={JOINTPRS_PHI}")


# ============================================================
# 4.4 Display analysis configuration
# ============================================================

print("=" * 65)
print("JointPRS ANALYSIS CONFIGURATION")
print("=" * 65)

print("Populations:", ", ".join(JOINTPRS_POPULATIONS))
print("GWAS sample sizes:", JOINTPRS_GWAS_N)
print("LD reference:", JOINTPRS_REF_DIR)
print("Target genotype:", JOINTPRS_BIM_PREFIX)

print("\nModel parameters:")
print("Chromosome:", JOINTPRS_CHROM)
print("Phi:", JOINTPRS_PHI)
print("Rho constraints:", JOINTPRS_RHO_CONS)
print("Prior a:", JOINTPRS_A)
print("Prior b:", JOINTPRS_B)
print("Meta-analysis:", JOINTPRS_META)

print("\nMCMC parameters:")
print("Iterations:", JOINTPRS_N_ITER)
print("Burn-in:", JOINTPRS_N_BURNIN)
print("Thinning:", JOINTPRS_THIN)
print("Seed:", JOINTPRS_SEED)

print("\nOutput directory:", JOINTPRS_OUTPUT)


# ============================================================
# 4.5 Run JointPRS
# ============================================================

run_env = os.environ.copy()

run_env.update({
    "OMP_NUM_THREADS": str(JOINTPRS_THREADS),
    "MKL_NUM_THREADS": str(JOINTPRS_THREADS),
    "OPENBLAS_NUM_THREADS": str(JOINTPRS_THREADS),
    "NUMEXPR_NUM_THREADS": str(JOINTPRS_THREADS),
})

print("\nRunning JointPRS...", flush=True)

subprocess.run(
    cmd,
    check=True,
    env=run_env
)

print("\nJointPRS completed.")


# ============================================================
# 4.6 Check posterior effect files
# ============================================================

phi_tag = (
    "phiauto"
    if JOINTPRS_PHI == "auto"
    else f"phi{float(JOINTPRS_PHI):.0e}"
)

effect_files_jointprs = {}

expected_populations = (
    JOINTPRS_POPULATIONS
    + (["META"] if JOINTPRS_META else [])
)

for population in expected_populations:

    filename = (
        f"{JOINTPRS_OUT_NAME}_{population}_pst_eff_"
        f"a{JOINTPRS_A:g}_b{JOINTPRS_B:.1f}_"
        f"{phi_tag}_chr{JOINTPRS_CHROM}.txt"
    )

    file = JOINTPRS_OUTPUT / filename

    assert file.is_file(), f"Missing effect file: {file}"

    effect_files_jointprs[population] = file

# Posterior genetic correlation matrix
corr_file = (
    JOINTPRS_OUTPUT /
    f"{JOINTPRS_OUT_NAME}_pst_corr_"
    f"a{JOINTPRS_A:g}_b{JOINTPRS_B:.1f}_"
    f"{phi_tag}_chr{JOINTPRS_CHROM}.txt"
)

assert corr_file.is_file(), (
    f"Missing correlation matrix: {corr_file}"
)

print("\n" + "=" * 65)
print("JointPRS POSTERIOR EFFECT FILES")
print("=" * 65)

for population, file in effect_files_jointprs.items():
    print(f"{population:5s}: {file.name}")

print("\nTotal effect files:", len(effect_files_jointprs))

# ============================================================
# 4.7 Estimated Genetic Correlation Matrix
# ============================================================

rho_matrix = np.loadtxt(corr_file)

assert rho_matrix.shape == (
    len(JOINTPRS_POPULATIONS),
    len(JOINTPRS_POPULATIONS)
)

print("\nEstimated genetic correlation matrix:")

display(
    pd.DataFrame(
        rho_matrix,
        index=JOINTPRS_POPULATIONS,
        columns=JOINTPRS_POPULATIONS
    ).round(4)
)

print("\nAll expected JointPRS outputs verified.")

JointPRS ANALYSIS CONFIGURATION
Populations: EAS, EUR, SAS
GWAS sample sizes: {'EAS': 283423, 'EUR': 933970, 'SAS': 49492}
LD reference: External_data/LD_reference
Target genotype: Demo_data/1-3.Sample_genotype/demo_chr22

Model parameters:
Chromosome: 22
Phi: 0.01
Rho constraints: [1, 1, 1]
Prior a: 1
Prior b: 0.5
Meta-analysis: True

MCMC parameters:
Iterations: 3000
Burn-in: 1500
Thinning: 5
Seed: 20261102

Output directory: Outputs/Step1_PRS/JointPRS/phi1e-2

Running JointPRS...


--ref_dir=External_data/LD_reference
--bim_prefix=Demo_data/1-3.Sample_genotype/demo_chr22
--sst_file=['External_data/GWAS/DIAMANTE_EAS.txt', 'External_data/GWAS/DIAMANTE_EUR.txt', 'External_data/GWAS/DIAMANTE_SAS.txt']
--rho_cons=[1.0, 1.0, 1.0]
--a=1.0
--b=0.5
--phi=0.01
--n_gwas=[283423, 933970, 49492]
--pop=['EAS', 'EUR', 'SAS']
--n_iter=3000
--n_burnin=1500
--thin=5
--out_dir=Outputs/Step1_PRS/JointPRS/phi1e-2
--out_name=Demo_JointPRS
--chrom=['22']
--meta=TRUE
--seed=20261102


*** 3 discovery popul

,EAS,EUR,SAS
EAS,1.0000,0.8260,0.6598
EUR,0.8260,1.0000,0.5369
SAS,0.6598,0.5369,1.0000



All expected JointPRS outputs verified.


In [8]:
# ============================================================
# 5. PLINK2: Individual-level PRS Scoring
# ============================================================

import sys
import subprocess
import pandas as pd
from pathlib import Path

# ============================================================
# 5.1 Configuration
# ============================================================

PLINK2 = Path(sys.prefix) / "bin/plink2"

SCORE_OUTPUT = OUTPUT_DIR / "PRS_scoring"
SCORE_OUTPUT.mkdir(parents=True, exist_ok=True)

SCORE_METHODS = {
    "PRSCSx": effect_files,
    "JointPRS": effect_files_jointprs,
}

SCORE_POPULATIONS = ["EAS", "EUR", "SAS", "META"]

assert PLINK2.is_file(), "PLINK2 not found in the current kernel."

for method, files in SCORE_METHODS.items():
    for pop in SCORE_POPULATIONS:
        assert pop in files, f"Missing {method} {pop} effects."
        assert files[pop].is_file(), f"Missing file: {files[pop]}"

# ============================================================
# 5.2 Calculate eight PRS scores
# ============================================================

prs_table = None

for method, files in SCORE_METHODS.items():
    for pop in SCORE_POPULATIONS:

        score_name = f"{method}_{pop}"
        output_prefix = SCORE_OUTPUT / score_name

        cmd = [
            str(PLINK2),
            "--bfile", str(GENOTYPE_PREFIX),
            "--score", str(files[pop]),
            "2", "4", "6",
            "cols=scoresums",
            "--out", str(output_prefix)
        ]

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True
        )

        # Keep full logs outside the Notebook
        log_file = Path(str(output_prefix) + ".run.log")
        log_file.write_text(
            result.stdout + "\n" + result.stderr,
            encoding="utf-8"
        )

        if result.returncode != 0:
            raise RuntimeError(
                f"PLINK2 scoring failed for {score_name}. "
                f"See {log_file.relative_to(PROJECT_ROOT)}"
            )

        # Load PLINK2 score output
        sscore_file = Path(str(output_prefix) + ".sscore")

        df = pd.read_csv(sscore_file, sep=r"\s+")
        df.columns = df.columns.str.lstrip("#")

        score_columns = [
            c for c in df.columns
            if c.startswith("SCORE") and c.endswith("_SUM")
        ]

        assert len(score_columns) == 1

        current = df[["IID", score_columns[0]]].copy()
        current.columns = ["Demo_ID", score_name]
        current["Demo_ID"] = current["Demo_ID"].astype(str)

        assert len(current) == 100
        assert current["Demo_ID"].is_unique

        if prs_table is None:
            prs_table = current
        else:
            prs_table = prs_table.merge(
                current,
                on="Demo_ID",
                how="inner",
                validate="one_to_one"
            )

        print(f"Completed: {score_name}")

# ============================================================
# 5.3 Save and summarize
# ============================================================

assert len(prs_table) == 100
assert prs_table.shape[1] == 9
assert not prs_table.isna().any().any()

prs_table = prs_table.sort_values("Demo_ID").reset_index(drop=True)

OUTPUT_FILE = SCORE_OUTPUT / "PRS_100_samples_8scores_SUM.csv"

prs_table.to_csv(OUTPUT_FILE, index=False)

print("\nPRS table:", prs_table.shape)
print("Output:", OUTPUT_FILE.relative_to(PROJECT_ROOT))

display(prs_table.head())

print("\nPRS distribution:")
display(
    prs_table.drop(columns="Demo_ID")
    .agg(["mean", "std", "min", "max"])
    .T
    .round(5)
)

Completed: PRSCSx_EAS
Completed: PRSCSx_EUR
Completed: PRSCSx_SAS
Completed: PRSCSx_META
Completed: JointPRS_EAS
Completed: JointPRS_EUR
Completed: JointPRS_SAS
Completed: JointPRS_META

PRS table: (100, 9)
Output: Outputs/Step1_PRS/PRS_scoring/PRS_100_samples_8scores_SUM.csv


,Demo_ID,PRSCSx_EAS,PRSCSx_EUR,PRSCSx_SAS,PRSCSx_META,JointPRS_EAS,JointPRS_EUR,JointPRS_SAS,JointPRS_META
0,D001,0.016251,0.013725,0.021142,0.010724,0.053765,0.035519,0.100509,0.038539
1,D002,0.024662,0.000069,0.014813,0.004446,0.049141,0.023558,0.094826,0.028778
2,D003,0.016753,-0.014030,0.032071,-0.005834,0.047893,0.015849,0.124657,0.022169
3,D004,0.021090,0.007756,0.030672,0.006368,0.052913,0.030316,0.128442,0.036160
4,D005,0.018364,-0.001820,0.012305,0.002918,0.027697,0.016102,0.068526,0.018392



PRS distribution:


,mean,std,min,max
PRSCSx_EAS,0.01830,0.00556,0.00292,0.03201
PRSCSx_EUR,0.00319,0.00641,-0.01403,0.01901
PRSCSx_SAS,0.02671,0.01137,0.00265,0.06231
PRSCSx_META,0.00498,0.00462,-0.00583,0.01401
JointPRS_EAS,0.05045,0.01839,0.00833,0.09864
JointPRS_EUR,0.02633,0.00948,0.00693,0.04564
JointPRS_SAS,0.10691,0.02946,0.04517,0.20241
JointPRS_META,0.03151,0.01092,0.00952,0.05461


In [9]:
# ============================================================
# 6. Merge PRS with Sample Information
# ============================================================

# Load participant information
sample_info = pd.read_csv(
    PROJECT_ROOT / "Demo_data/1-1.Sample_information/sample_information.csv",
    dtype={"Demo_ID": str}
)

# Merge all eight PRS scores
prs_dataset = sample_info.merge(
    prs_table,
    on="Demo_ID",
    how="left",
    validate="one_to_one"
)

# Confirm all participants have PRS
prs_columns = [c for c in prs_table.columns if c != "Demo_ID"]

assert len(prs_dataset) == 100
assert prs_dataset["Demo_ID"].is_unique
assert prs_dataset[prs_columns].notna().all().all()

# Save integrated dataset
MERGED_FILE = OUTPUT_DIR / "PRS_100_samples_with_information.csv"

prs_dataset.to_csv(MERGED_FILE, index=False)

print("Participants:", len(prs_dataset))
print("PRS features:", len(prs_columns))
print("Missing PRS values:", prs_dataset[prs_columns].isna().sum().sum())
print("Output:", MERGED_FILE.relative_to(PROJECT_ROOT))

display(prs_dataset.head())

Participants: 100
PRS features: 8
Missing PRS values: 0
Output: Outputs/Step1_PRS/PRS_100_samples_with_information.csv


,Demo_ID,T2D,Dataset,AGE,SEX,Familyall_numeric,N_ABD,N_BMD_left_hip,N_BMD_right_hip,N_BMD_spine,PRSCSx_EAS,PRSCSx_EUR,PRSCSx_SAS,PRSCSx_META,JointPRS_EAS,JointPRS_EUR,JointPRS_SAS,JointPRS_META
0,D001,0,training2,65,2,0,21,1,1,1,0.016251,0.013725,0.021142,0.010724,0.053765,0.035519,0.100509,0.038539
1,D002,0,testing,75,1,0,27,1,1,1,0.024662,0.000069,0.014813,0.004446,0.049141,0.023558,0.094826,0.028778
2,D003,1,training2,54,2,0,23,1,1,1,0.016753,-0.014030,0.032071,-0.005834,0.047893,0.015849,0.124657,0.022169
3,D004,1,training1,61,1,0,23,1,1,1,0.021090,0.007756,0.030672,0.006368,0.052913,0.030316,0.128442,0.036160
4,D005,1,training2,73,1,0,24,1,1,1,0.018364,-0.001820,0.012305,0.002918,0.027697,0.016102,0.068526,0.018392
